# Fitting a flow to unnormalized data

A `fleqx` flow's base distribution starts at `loc=0, scale=1`, and its conditioner networks are initialized near-identity. If your data is nowhere near that scale (e.g. large offsets, or wildly different per-dimension scales), training can struggle badly -- the flow has to learn its way from `N(0, 1)` to your data's actual location/scale before it can start fitting the shape at all.

This notebook demonstrates the problem on synthetic "unnormalized" data, and shows the fix: passing `data=` to a flow constructor (`coupling_flow`, `masked_autoregressive_flow`, `planar_flow`), which prepends a fixed-shape `ScalarAffine` initialized from that data's mean/std.

Requires `matplotlib` in addition to `fleqx` (`pip install matplotlib`).

In [ ]:
import jax
import jax.numpy as jnp
import jax.random as jr
import matplotlib.pyplot as plt
import parax

import fleqx

## 1. Generate unnormalized synthetic data

A two-moons dataset (a standard flow benchmark, since it isn't just a Gaussian a diagonal
affine could fit on its own), shifted and rescaled to look like "real world" unnormalized
data: a large offset, and very different scales across the two dimensions.

In [ ]:
def make_moons(key, n, noise=0.05):
    key_theta, key_noise = jr.split(key)
    n_half = n // 2
    theta = jnp.linspace(0, jnp.pi, n_half)
    moon0 = jnp.stack([jnp.cos(theta), jnp.sin(theta)], axis=1)
    moon1 = jnp.stack([1 - jnp.cos(theta), 0.5 - jnp.sin(theta)], axis=1)
    data = jnp.concatenate([moon0, moon1], axis=0)
    return data + noise * jr.normal(key_noise, data.shape)


# A large offset on dim 0, and very different per-dimension scales -- the kind of
# thing that shows up with real, un-preprocessed data.
SHIFT = jnp.array([500.0, -250.0])
SCALE = jnp.array([80.0, 5.0])


def make_unnormalized_moons(key, n, noise=0.05):
    return make_moons(key, n, noise=noise) * SCALE + SHIFT


DIM = 2
key = jr.key(0)
data_key, flow_key, fit_key, sample_key = jr.split(key, 4)

train_data = make_unnormalized_moons(data_key, 4000)
print("mean:", train_data.mean(axis=0), " std:", train_data.std(axis=0))

plt.figure(figsize=(4, 4))
plt.scatter(train_data[:, 0], train_data[:, 1], s=3)
plt.title("Unnormalized synthetic data")
plt.show()

## 2. Build two identical flows, one with `data=` and one without

Both flows are built from the *same* `flow_key`, so their conditioner networks start
with identical weights -- `data=` doesn't consume randomness, it just prepends a
standardizing bijector. That isolates the comparison to exactly the thing we're testing.

In [ ]:
FLOW_KWARGS = dict(dim=DIM, flow_layers=4, nn_width=32)

flow_raw = fleqx.flows.coupling_flow(flow_key, **FLOW_KWARGS)
flow_std = fleqx.flows.coupling_flow(flow_key, **FLOW_KWARGS, data=train_data)

## 3. Fit both on the same data

Same `fit_key` for both calls too, so batching/shuffling is identical between runs.

In [ ]:
FIT_KWARGS = dict(learning_rate=1e-3, max_epochs=150, batch_size=256, show_progress=False)

trained_raw, losses_raw = fleqx.train.fit(fit_key, flow_raw, train_data, **FIT_KWARGS)
trained_std, losses_std = fleqx.train.fit(fit_key, flow_std, train_data, **FIT_KWARGS)

print(f"final train NLL, no data=  : {losses_raw['train'][-1]:.1f}")
print(f"final train NLL, data=     : {losses_std['train'][-1]:.1f}")

## 4. Compare training curves

Separate subplots, since the two losses are on wildly different scales: the
unstandardized flow spends its budget crawling from `N(0, 1)` towards `data`'s
actual location, and 150 epochs isn't nearly enough -- Adam's step size is roughly
bounded per-parameter regardless of gradient size, so closing a gap of hundreds of
units takes far longer than closing a gap of order 1.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, losses, title in [
    (axes[0], losses_raw, "no data= (raw scale)"),
    (axes[1], losses_std, "data= (standardized)"),
]:
    ax.plot(losses["train"], label="train")
    ax.plot(losses["val"], label="val")
    ax.set_xlabel("epoch")
    ax.set_ylabel("negative log-likelihood")
    ax.set_title(title)
    ax.legend()
plt.tight_layout()
plt.show()

## 5. Compare samples visually

In [ ]:
sample_keys = jr.split(sample_key, 2000)
samples_raw = jax.vmap(trained_raw.sample)(sample_keys)
samples_std = jax.vmap(trained_std.sample)(sample_keys)

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharex=True, sharey=True)
for ax, samples, title in [
    (axes[0], train_data, "true data"),
    (axes[1], samples_raw, "flow samples, no data="),
    (axes[2], samples_std, "flow samples, data="),
]:
    ax.scatter(samples[:, 0], samples[:, 1], s=3)
    ax.set_title(title)
plt.tight_layout()
plt.show()

## Bonus: the standardizing affine stays fixed

`data=`'s `ScalarAffine` is initialized from `train_data`'s mean/std and stays
exactly there -- fleqx wraps it in `parax.Freeze`, which `fit` resolves (via
`parax.unwrap`, applying `jax.lax.stop_gradient`) fresh on every call rather than
letting gradient descent touch it. That matters more than it might sound: `ScalarAffine`
stores `scale`, `inv_scale` and `log_scale` as independent leaves that are supposed
to satisfy `scale == 1/inv_scale == exp(log_scale)`, but MLE training only ever
exercises the inverse direction (`forward` is only used by `sample`), which touches
`inv_scale`/`log_scale` but never `scale`. Without freezing it, gradient descent
would drift the two it does touch and desynchronize all three, silently breaking
`forward(inverse(y)) == y`. This pokes at `flow_std`'s internal structure purely to
illustrate that; it isn't part of the public API.

In [ ]:
initial_affine = parax.unwrap(flow_std.bijector.bijectors[0].bijector)
trained_affine = parax.unwrap(trained_std.bijector.bijectors[0].bijector)

print("shift  before:", initial_affine.shift, " after:", trained_affine.shift)
print("scale  before:", initial_affine.scale, " after:", trained_affine.scale)

# The affine is still a true bijection after training, since it was never touched.
y = train_data[0]
roundtrip_error = trained_affine.forward(trained_affine.inverse(y)) - y
print("round-trip error:", roundtrip_error)